<a href="https://colab.research.google.com/github/hussain1079/Assessment3/blob/main/Assessment3_Muhammed_Hussain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install catboost -q

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

train_df = pd.read_csv('/content/drive/MyDrive/DSA ICT/Assessment 3/train_LZdllcl.csv')
test_df = pd.read_csv('/content/drive/MyDrive/DSA ICT/Assessment 3/test_2umaH9m.csv')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

Train shape: (54808, 14)
Test shape: (23490, 13)


In [6]:
# Fill missing values before the train/validation split
train_df['education'] = train_df['education'].fillna(train_df['education'].mode()[0])
train_df['previous_year_rating'] = train_df['previous_year_rating'].fillna(train_df['previous_year_rating'].median())

print('Remaining missing values:')
print(train_df.isnull().sum())

Remaining missing values:
employee_id             0
department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
KPIs_met >80%           0
awards_won?             0
avg_training_score      0
is_promoted             0
dtype: int64


In [7]:
X = train_df.drop(columns=['is_promoted', 'employee_id'])
y = train_df['is_promoted']
X_test = test_df.drop(columns=['employee_id'])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
numerical_columns = X.select_dtypes(exclude=['object']).columns.tolist()

print('Training:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Training: (43846, 12)
Validation: (10962, 12)
Test: (23490, 12)


In [8]:
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numerical_columns),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_columns)
])

X_train_final = preprocessor.fit_transform(X_train)
X_val_final = preprocessor.transform(X_val)

print('Encoded training:', X_train_final.shape)
print('Encoded validation:', X_val_final.shape)

Encoded training: (43846, 58)
Encoded validation: (10962, 58)


In [9]:
# XGBoost
xgb_model = XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85, min_child_weight=2,
    objective='binary:logistic', eval_metric='logloss',
    random_state=42, n_jobs=-1
)

xgb_model.fit(X_train_final, y_train)
xgb_val_probability = xgb_model.predict_proba(X_val_final)[:, 1]

xgb_thresholds = np.arange(0.20, 0.60, 0.001)
xgb_f1_scores = []
for threshold in xgb_thresholds:
    xgb_f1_scores.append(f1_score(y_val, (xgb_val_probability >= threshold).astype(int)))

xgb_best_index = np.argmax(xgb_f1_scores)
xgb_best_threshold = xgb_thresholds[xgb_best_index]
xgb_best_f1 = xgb_f1_scores[xgb_best_index]

print('XGBoost Best Threshold:', xgb_best_threshold)
print('XGBoost Best F1:', xgb_best_f1)

XGBoost Best Threshold: 0.2890000000000001
XGBoost Best F1: 0.540222367560497


In [10]:
# CatBoost
cat_train = X_train.copy()
cat_val = X_val.copy()
for col in categorical_columns:
    cat_train[col] = cat_train[col].fillna('Missing')
    cat_val[col] = cat_val[col].fillna('Missing')

cat_model = CatBoostClassifier(
    iterations=600, depth=6, learning_rate=0.05,
    loss_function='Logloss', cat_features=categorical_columns,
    class_weights=[1, 3.5], l2_leaf_reg=5,
    random_seed=42, verbose=False
)
cat_model.fit(cat_train, y_train, eval_set=(cat_val, y_val), verbose=False)
cat_val_probability = cat_model.predict_proba(cat_val)[:, 1]

print('CatBoost validation probability mean:', cat_val_probability.mean())

CatBoost validation probability mean: 0.1684635491715281


In [11]:
# 50/50 XGBoost + CatBoost validation blend
blend_probability = 0.5 * xgb_val_probability + 0.5 * cat_val_probability
blend_thresholds = np.arange(0.25, 0.61, 0.01)
blend_f1_scores = []
for threshold in blend_thresholds:
    blend_f1_scores.append(f1_score(y_val, (blend_probability >= threshold).astype(int)))

blend_best_index = np.argmax(blend_f1_scores)
blend_best_threshold = blend_thresholds[blend_best_index]
blend_best_f1 = blend_f1_scores[blend_best_index]

print('50/50 Blend Best Threshold:', blend_best_threshold)
print('50/50 Blend Best F1:', blend_best_f1)

50/50 Blend Best Threshold: 0.4600000000000002
50/50 Blend Best F1: 0.5366197183098591


In [12]:
# Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=3,
    min_samples_split=10, min_samples_leaf=5, random_state=42
)
gb_model.fit(X_train_final, y_train)
gb_val_probability = gb_model.predict_proba(X_val_final)[:, 1]

# Candidate A validation check
candidate_a_val_probability = (
    0.35 * xgb_val_probability +
    0.45 * cat_val_probability +
    0.20 * gb_val_probability
)
candidate_a_val_prediction = (candidate_a_val_probability >= 0.37).astype(int)
print('Candidate A validation F1:', f1_score(y_val, candidate_a_val_prediction))

Candidate A validation F1: 0.5296969696969697


In [13]:
# Final XGBoost on all training rows
xgb_final = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85, min_child_weight=2,
        objective='binary:logistic', eval_metric='logloss',
        random_state=42, n_jobs=-1
    ))
])
xgb_final.fit(X, y)
xgb_test_probability = xgb_final.predict_proba(X_test)[:, 1]
print('Final XGBoost ready.')

Final XGBoost ready.


In [14]:
# Final CatBoost on all training rows
cat_full = X.copy()
cat_test_full = X_test.copy()
for col in categorical_columns:
    cat_full[col] = cat_full[col].fillna('Missing')
    cat_test_full[col] = cat_test_full[col].fillna('Missing')

cat_final = CatBoostClassifier(
    iterations=600, depth=6, learning_rate=0.05,
    loss_function='Logloss', cat_features=categorical_columns,
    class_weights=[1, 3.5], l2_leaf_reg=5,
    random_seed=42, verbose=False
)
cat_final.fit(cat_full, y, verbose=False)
cat_test_probability = cat_final.predict_proba(cat_test_full)[:, 1]
print('Final CatBoost ready.')

Final CatBoost ready.


In [15]:
# Final Gradient Boosting
# Keep the same preprocessing structure, refit it on all training data for the final model.
X_full_final = preprocessor.fit_transform(X)
X_test_full_final = preprocessor.transform(X_test)

gb_final = GradientBoostingClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=3,
    min_samples_split=10, min_samples_leaf=5, random_state=42
)
gb_final.fit(X_full_final, y)
gb_test_probability = gb_final.predict_proba(X_test_full_final)[:, 1]
print('Final Gradient Boosting ready.')

Final Gradient Boosting ready.


In [16]:
# Final blend
final_probability = (
    0.35 * xgb_test_probability +
    0.45 * cat_test_probability +
    0.20 * gb_test_probability
)

final_prediction = (final_probability >= 0.37).astype(int)

print('Predicted promotions:', int(final_prediction.sum()))
print('Not promoted:', int((final_prediction == 0).sum()))
print('Promotion rate:', final_prediction.mean())

Predicted promotions: 1306
Not promoted: 22184
Promotion rate: 0.05559812686249468


In [18]:
# Create submission
submission = pd.DataFrame({
    'employee_id': test_df['employee_id'],
    'is_promoted': final_prediction
})

submission.to_csv('/content/drive/MyDrive/DSA ICT/Assessment 3/submission.csv', index=False)

print('Submission saved to: submission.csv')
print('Shape:', submission.shape)
print(submission['is_promoted'].value_counts())

Submission saved to: submission.csv
Shape: (23490, 2)
is_promoted
0    22184
1     1306
Name: count, dtype: int64


In [19]:
# Final validation
print('Columns:', submission.columns.tolist())
print('Rows:', len(submission))
print('Unique employee IDs:', submission['employee_id'].nunique())
print('Missing values:')
print(submission.isnull().sum())
print('First 5 rows:')
display(submission.head())

Columns: ['employee_id', 'is_promoted']
Rows: 23490
Unique employee IDs: 23490
Missing values:
employee_id    0
is_promoted    0
dtype: int64
First 5 rows:


,employee_id,is_promoted
0,8724,0
1,74430,0
2,72255,0
3,38562,0
4,64486,0
